#### Librerias

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
from scipy import sparse
import joblib
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from scipy.stats import uniform
import matplotlib.pyplot as plt
import pandas as pd

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#### Variables

In [ ]:
MODELOS_PATH = "/content/drive/MyDrive/Trabajo práctico 3/modelos"

#### Carga de datos

In [ ]:
X_train_bow = sparse.load_npz(f"{MODELOS_PATH}/X_train_bow.npz")
X_train_tfidf = sparse.load_npz(f"{MODELOS_PATH}/X_train_tfidf.npz")
X_test_bow = sparse.load_npz(f"{MODELOS_PATH}/X_test_bow.npz")
X_test_tfidf = sparse.load_npz(f"{MODELOS_PATH}/X_test_tfidf.npz")

y_train = np.load(f"{MODELOS_PATH}/y_train.npy")
y_test = np.load(f"{MODELOS_PATH}/y_test.npy")

#### Subset binario de test

In [ ]:
mask_binario = y_test != 2

X_test_bow_bin = X_test_bow[mask_binario]
X_test_tfidf_bin = X_test_tfidf[mask_binario]
y_test_bin = y_test[mask_binario]

#### Muestra estratificada para la búsqueda de hiperparámetros

In [ ]:
idx_muestra, _ = train_test_split(
    np.arange(len(y_train)), train_size=200_000, stratify=y_train, random_state=42
)

X_muestra_bow = X_train_bow[idx_muestra]
X_muestra_tfidf = X_train_tfidf[idx_muestra]
y_muestra = y_train[idx_muestra]

La búsqueda de hiperparámetros se hace sobre 200 mil tweets (muestra estratificada,
mantiene el 50/50) y no sobre los 1.6M, por una cuestión de tiempo de cómputo:
RandomizedSearchCV entrena cada combinación 3 veces (cross-validation), lo que
multiplica el costo. Importante: esto es solo para BUSCAR los mejores
hiperparámetros — el modelo final se reentrena con el training completo.

#### Optimización — Naive Bayes

In [ ]:
param_dist_nb = {"alpha": uniform(loc=0.01, scale=2.0)}

random_search_nb = RandomizedSearchCV(
    MultinomialNB(),
    param_distributions=param_dist_nb,
    n_iter=8,
    cv=3,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

random_search_nb.fit(X_muestra_bow, y_muestra)
random_search_nb.best_params_

{'alpha': np.float64(1.9114286128198323)}

El mejor alpha encontrado fue 1.91, casi el doble del default (1.0) — más suavizado
del que trae de fábrica. Aun así, como se ve en el reporte de abajo, el accuracy en
test quedó igual al baseline (0.83): el ajuste existe pero no mueve la aguja.

#### Reentrenamiento de NB con el mejor alpha, sobre todo el training

In [ ]:
mejor_alpha = random_search_nb.best_params_["alpha"]
modelo_nb_opt = MultinomialNB(alpha=mejor_alpha)
modelo_nb_opt.fit(X_train_bow, y_train)

pred_nb_opt = modelo_nb_opt.predict(X_test_bow_bin)
print(classification_report(y_test_bin, pred_nb_opt, labels=[0, 4], target_names=["Negativo", "Positivo"]))

              precision    recall  f1-score   support

    Negativo       0.83      0.84      0.83       177
    Positivo       0.84      0.83      0.83       182

    accuracy                           0.83       359
   macro avg       0.83      0.83      0.83       359
weighted avg       0.83      0.83      0.83       359



#### Optimización — Logistic Regression (C)

In [ ]:
param_dist_lr = {"C": uniform(loc=0.01, scale=10)}

random_search_lr = RandomizedSearchCV(
    LogisticRegression(max_iter=1000),
    param_distributions=param_dist_lr,
    n_iter=8,
    cv=3,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

random_search_lr.fit(X_muestra_tfidf, y_muestra)
random_search_lr.best_params_

{'C': np.float64(0.5908361216819946)}

Para LR el mejor C fue 0.59, menor al default (1.0), o sea más regularización. Igual
que con NB, el accuracy final no cambió (0.82). La lectura conjunta: con 1.6M de
ejemplos balanceados, los modelos ya generalizaban bien con su configuración por
defecto — el techo de performance de este proyecto no estaba en los hiperparámetros.

#### Reentrenamiento de LR con el mejor C, sobre todo el training

In [ ]:
mejor_c = random_search_lr.best_params_["C"]
modelo_lr_opt = LogisticRegression(C=mejor_c, max_iter=1000)
modelo_lr_opt.fit(X_train_tfidf, y_train)

pred_lr_opt = modelo_lr_opt.predict(X_test_tfidf_bin)
print(classification_report(y_test_bin, pred_lr_opt, labels=[0, 4], target_names=["Negativo", "Positivo"]))

              precision    recall  f1-score   support

    Negativo       0.82      0.81      0.81       177
    Positivo       0.82      0.83      0.82       182

    accuracy                           0.82       359
   macro avg       0.82      0.82      0.82       359
weighted avg       0.82      0.82      0.82       359



#### Comparación: baseline vs optimizado

In [ ]:
modelo_nb_base = joblib.load(f"{MODELOS_PATH}/modelo_nb_baseline.pkl")
modelo_lr_base = joblib.load(f"{MODELOS_PATH}/modelo_lr_baseline.pkl")

acc_nb_base = accuracy_score(y_test_bin, modelo_nb_base.predict(X_test_bow_bin))
acc_lr_base = accuracy_score(y_test_bin, modelo_lr_base.predict(X_test_tfidf_bin))
acc_nb_opt = accuracy_score(y_test_bin, pred_nb_opt)
acc_lr_opt = accuracy_score(y_test_bin, pred_lr_opt)

tabla = pd.DataFrame({
    "modelo": ["Naive Bayes", "Naive Bayes", "Logistic Regression", "Logistic Regression"],
    "version": ["baseline", "optimizado", "baseline", "optimizado"],
    "accuracy": [acc_nb_base, acc_nb_opt, acc_lr_base, acc_lr_opt]
})
tabla

,modelo,version,accuracy
0,Naive Bayes,baseline,0.827298
1,Naive Bayes,optimizado,0.832869
2,Logistic Regression,baseline,0.816156
3,Logistic Regression,optimizado,0.818942


#### Chequeo de overfitting: accuracy en train vs test

In [ ]:
pred_nb_opt_train = modelo_nb_opt.predict(X_train_bow)
pred_lr_opt_train = modelo_lr_opt.predict(X_train_tfidf)

acc_nb_train = accuracy_score(y_train, pred_nb_opt_train)
acc_lr_train = accuracy_score(y_train, pred_lr_opt_train)

acc_nb_test = accuracy_score(y_test_bin, pred_nb_opt)
acc_lr_test = accuracy_score(y_test_bin, pred_lr_opt)

print(f"Naive Bayes  — train: {acc_nb_train:.3f} | test: {acc_nb_test:.3f} | diferencia: {acc_nb_train - acc_nb_test:.3f}")
print(f"Logistic Reg — train: {acc_lr_train:.3f} | test: {acc_lr_test:.3f} | diferencia: {acc_lr_train - acc_lr_test:.3f}")

Naive Bayes  — train: 0.774 | test: 0.833 | diferencia: -0.059
Logistic Reg — train: 0.787 | test: 0.819 | diferencia: -0.032


Resultado llamativo: el accuracy es MEJOR en test que en train para los dos modelos
(NB: 0.774 vs 0.833; LR: 0.787 vs 0.819). No hay overfitting — es lo contrario de
lo que uno esperaría. La explicación más probable está en el origen de las
etiquetas: las del training se asignaron automáticamente según el emoticono del
tweet (método ruidoso — no todo tweet con ":)" es realmente positivo), mientras que
las del test las puso una persona a mano. Parte de los "errores" en train son en
realidad casos donde el modelo predijo bien pero la etiqueta automática estaba mal.

#### Guardado de modelos optimizados

In [ ]:
joblib.dump(modelo_nb_opt, f"{MODELOS_PATH}/modelo_nb_optimizado.pkl")
joblib.dump(modelo_lr_opt, f"{MODELOS_PATH}/modelo_lr_optimizado.pkl")

['/content/drive/MyDrive/Trabajo práctico 3/modelos/modelo_lr_optimizado.pkl']